# The `SspPrior` data structure, and fusing independent priors with `combine_states_priors`

This chapter is two short, mostly-independent demos rather than a staged build like `ssp_01`...`ssp_08`:

1. **What `SspPrior` actually is** -- the contract it validates, the read-only facade it exposes, and
   the two gotchas its own docstring flags (`xr.merge` rejects it; it doesn't survive xarray ops).
2. **`combine_states_priors(prior_a, prior_b)`** -- inverse-variance fusion of two *independent* priors
   over the same states, e.g. a vendor study and an internal panel that each disclose the same media
   channels over different windows with different confidence.

Both pieces live in [`bunobee.models.ssp.prior`](../../src/bunobee/models/ssp/prior.py). Reading
`ssp_02_time_point_priors.ipynb` first helps (it introduces time-point priors), but nothing here
strictly depends on `ssp_01`...`ssp_08`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from jax import numpy as jnp

from bunobee.models.ssp.plotting import plot_prior_heatmap, plot_states
from bunobee.models.ssp.prior import (
    SspPrior,
    combine_states_priors,
    extend_states_prior_nearest,
    extend_states_prior_smoothed,
)
from bunobee.simulation.ssp import construct_states_prior


## 1. The `SspPrior` data structure

`SspPrior` is a frozen wrapper around a *complete*, filter-ready prior `xr.Dataset`. "Complete" means
five things are present and shaped correctly:

- `a0`, `P0` -- initial-state mean / variance over `(state,)` (`P0` diagonal here; a full
  `(state, state_dual)` covariance is also legal for `SspPrior` itself, though `combine_states_priors`
  below only accepts the diagonal form).
- `a_obs`, `P_obs` -- disclosed-state mean / variance over `(time, state)`, `P_obs = inf` marking an
  undisclosed step.
- `positivity` -- a boolean mask over `(state,)`.
- `time` and `state` as actual coordinates, not just bare dimensions.

Validation (`_validate`, wrapping the shared `validate_prior` contract) runs once, at construction
time, regardless of how the dataset was built -- by hand, by `construct_states_prior`, or loaded from
disk.

> **⚠️ Tip -- the schema doesn't encode *scale*.** All five fields above describe the **natural
> (linear) scale**, but `transform_to_ekf` / `transform_to_ekf_st` produce an **a-space** dataset with
> the identical shape and variable names for the EKF filters -- `SspPrior` cannot tell the two apart.
> That is exactly why those transforms deliberately hand back a bare `xr.Dataset`, never re-wrapping
> the result as `SspPrior`, even when given one. Section 2 below revisits this: fusing priors is only
> valid on the scale the evidence was actually observed on.


In [ ]:
# A minimal, hand-built complete prior: 5 steps, 2 states ("level", "tv").
n_steps, n_states = 5, 2

ds = xr.Dataset(
    {
        "a0": (("state",), np.array([0.0, 0.5])),
        "P0": (("state",), np.array([1.0, 0.8])),
        "a_obs": (("time", "state"), np.full((n_steps, n_states), np.nan)),
        "P_obs": (("time", "state"), np.full((n_steps, n_states), np.inf)),  # nothing disclosed yet
        "positivity": (("state",), np.array([False, True])),
    },
    coords={"time": np.arange(n_steps), "state": ["level", "tv"]},
)
# Disclose one anchor for "tv" at t=2.
ds["a_obs"][2, 1] = 0.6
ds["P_obs"][2, 1] = 0.05

prior = SspPrior(ds)
prior


### 1.1 Read access is a thin facade over the wrapped dataset

`prior["a0"]`, `prior.sizes`, `"time" in prior`, `len(prior)`, and iteration all delegate straight to
`prior.dataset` -- promoting a dataset to `SspPrior` costs nothing at the call site.


In [ ]:
print("prior['a0']       ->", prior["a0"].values)
print("prior.sizes       ->", dict(prior.sizes))
print("'time' in prior   ->", "time" in prior)  # __contains__ tests coords too, not just data vars
print("len(prior)        ->", len(prior))  # number of data vars: a0, P0, a_obs, P_obs, positivity
print("list(prior)       ->", list(prior))  # __iter__ yields data-var names, like xr.Dataset


### 1.2 The contract is enforced at construction time

Dropping any required variable -- here `a0` -- raises immediately, with a message naming exactly what's
missing, rather than surfacing as a cryptic failure deep inside a filter scan.


In [ ]:
incomplete = ds.drop_vars("a0")
try:
    SspPrior(incomplete)
except ValueError as exc:
    print("ValueError:", exc)


### 1.3 Two gotchas the class docstring calls out

**`xr.merge` rejects the wrapper outright.** Merging is the usual way to assemble a prior
(`xr.merge([base_prior, states_prior])`), so a wrapped prior must be unwrapped with `.to_dataset()`
first.


In [ ]:
try:
    xr.merge([prior, ds])
except TypeError as exc:
    print("TypeError:", exc)

# The escape hatch: unwrap first.
merged = xr.merge([prior.to_dataset(), xr.Dataset({"extra_note": 1})])
print("merge after to_dataset() ok:", "extra_note" in merged)


**The wrapper does not survive xarray operations.** `.copy()`, `.sel()`, `.isel()`, `.assign()`, and
netCDF/zarr round-trips all return a bare `xr.Dataset`, silently. `SspPrior` is a checkpoint that a
dataset was valid *when it was promoted*, not a durable guarantee that everything derived from it still
is -- re-promote with `SspPrior.from_dataset(...)` after any such operation.


In [ ]:
sub = prior.dataset.isel(time=slice(0, 3))
print("type after .isel():", type(sub).__name__)  # bare Dataset, not SspPrior

re_promoted = SspPrior.from_dataset(sub)
print("type after re-promotion:", type(re_promoted).__name__)


### 1.4 A realistic `SspPrior`, assembled the way production code does

`construct_states_prior` (simulation-only) builds the *disclosure* block -- `a_obs` / `P_obs` /
`positivity` -- but deliberately carries no `a0` / `P0`: the initial-state prior is supplied downstream
from real belief, not simulated. So its raw output is **not** a complete prior yet.


In [ ]:
regressors = ["tv", "search", "social"]
state_labels = ["level", *regressors]
n_states = len(state_labels)
n_steps = 52
dates = pd.date_range("2024-01-01", periods=n_steps, freq="W")

disclosure_only = construct_states_prior(
    n_steps=n_steps,
    n_states=n_states,
    true_states=jnp.array([0.0, 0.8, 0.5, 0.3]),  # level ignored (var stays inf)
    regressors=regressors,
    n_periods=3,
    n_points=4,
    seed=7,
    obs_scale=0.15,
)

try:
    SspPrior(disclosure_only)
except ValueError as exc:
    print("ValueError (expected -- no a0 / P0 yet):", exc)


`extend_states_prior_nearest` (see `ssp_07_extend_states_prior.ipynb`) fills the undisclosed steps
along the random-walk marginal *and* derives `a0` / `P0` from that same marginal at `t = -1` -- so its
return value is a complete, validated `SspPrior` by construction.

`extend_states_prior_smoothed` (see `ssp_08_multi_anchor_prior_extension.ipynb`) fills the
same undisclosed steps via an exact KF-forward + RTS-backward pass instead, fusing *every* anchor
in a channel rather than snapping to the nearest one -- the two agree for a single-anchor channel
and `_smoothed` is strictly tighter once a channel carries two or more. This notebook builds
`vendor` / `panel` from **both** methods on every run and compares them side by side below, rather
than picking one.


In [ ]:
vendor = extend_states_prior_nearest(disclosure_only, Q=0.02)
vendor_smoothed = extend_states_prior_smoothed(disclosure_only, Q=0.02)
print(type(vendor).__name__, "/", type(vendor_smoothed).__name__)
print("a0 (nearest):  ", dict(zip(state_labels, vendor.a0.values.round(3))))
print("a0 (smoothed): ", dict(zip(state_labels, vendor_smoothed.a0.values.round(3))))
print(
    "finite fraction of P_obs (nearest / smoothed):",
    np.isfinite(vendor.P_obs.values).mean().round(3),
    "/",
    np.isfinite(vendor_smoothed.P_obs.values).mean().round(3),
)

fig, axes = plot_prior_heatmap(vendor, quantity="both", dates=dates.values, title="a complete SspPrior")
plt.show()


## 2. `combine_states_priors` -- fusing independent evidence

Two sources of evidence about the *same* latent states -- a vendor study and an internal panel, two
separately-extended anchor sets -- can be combined into one prior by inverse-variance weighting, exact
for independent Gaussian evidence:

$$P = \left(P_1^{-1} + P_2^{-1}\right)^{-1}, \qquad a = P\left(P_1^{-1} a_1 + P_2^{-1} a_2\right).$$

Precision adds, so the fused variance is never wider than either input. The rule is applied elementwise
to `a_obs` / `P_obs` over `(time, state)` and to `a0` / `P0` over `(state,)`. Only those four moment
variables are fused: `positivity` must match exactly on both operands, and everything else (`sdy`, any
`sigma_q` block, other data vars, attrs) is taken from the **left** operand -- `prior_b`'s copies are
dropped.

> **⚠️ Linear scale only.** `combine_states_priors` (and, more broadly, everything in this notebook)
> operates on the **natural (linear) scale** -- the same scale `construct_states_prior` and
> `extend_states_prior_*` produce. Nothing about `SspPrior` or its schema distinguishes that from
> **a-space**, the log/multiplicative reparameterisation `transform_to_ekf` / `transform_to_ekf_st`
> produce for the EKF filters (`kalman_1d_ekf`, `kalman_1d_ekf_st`; see `ssp_03`/`ssp_05`) -- both
> share the exact same `a0`/`P0`/`a_obs`/`P_obs`/`positivity` shape, just with different semantics.
> That is precisely why the EKF transforms deliberately return a **bare `xr.Dataset`**, never an
> `SspPrior`, even when handed one: nothing stops you from feeding an a-space dataset into
> `combine_states_priors` and getting a numerically valid-looking answer back, but averaging precisions
> is only the correct posterior for the (linear) scale the evidence was actually observed on. Because
> `a = exp(k * a-space)` is nonlinear, fusing then transforming and transforming then fusing are **not**
> the same operation. **Always fuse natural-scale priors first, then call `transform_to_ekf(...)` /
> `transform_to_ekf_st(...)` on the fused result -- never the other way around.**


### 2.1 Two independent studies over the same channels

`vendor` (from section 1.4) discloses `tv` / `search` / `social` over three tight windows
(`obs_scale=0.15`). `panel` is a second, independent read on the *same* channels -- different windows
(a different seed), looser confidence (`obs_scale=0.25`) -- extended the same way, via **both**
`extend_states_prior_nearest` and `extend_states_prior_smoothed`, compared side by side below. `level`
is disclosed nowhere in either study, so it stays fully undisclosed straight through every fusion.


In [ ]:
panel_raw = construct_states_prior(
    n_steps=n_steps,
    n_states=n_states,
    true_states=jnp.array([0.0, 0.75, 0.55, 0.25]),
    regressors=regressors,
    n_periods=3,
    n_points=4,
    seed=99,
    obs_scale=0.25,  # looser than the vendor study
)
panel = extend_states_prior_nearest(panel_raw, Q=0.02)
panel_smoothed = extend_states_prior_smoothed(panel_raw, Q=0.02)

fused = combine_states_priors(vendor, panel)
fused_smoothed = combine_states_priors(vendor_smoothed, panel_smoothed)
print(type(fused).__name__, "/", type(fused_smoothed).__name__)

# No collapse toward zero in the smoothed extension anywhere (issue #69 regression), including the
# pre-first-anchor region of every anchored channel -- checked on every run of this notebook, not
# just in the isolated unit tests that motivated the fix.
for p in (vendor_smoothed, panel_smoothed):
    finite_p = np.isfinite(p.P_obs.values)
    assert not np.isnan(p.a_obs.values[finite_p]).any()
    assert (p.P_obs.values[finite_p] > 1e-6).all()  # not collapsed toward 0
finite_fused = np.isfinite(fused_smoothed.P_obs.values)
assert finite_fused[:, 1:].all()  # tv / search / social: disclosed by at least one study everywhere
assert np.isinf(fused_smoothed.P_obs.values[:, 0]).all()  # level: disclosed by neither study
# Precision adds: the fused variance is never wider than either input, at every finite step.
assert np.all(fused_smoothed.P_obs.values[finite_fused] <= vendor_smoothed.P_obs.values[finite_fused] + 1e-6)
assert np.all(fused_smoothed.P_obs.values[finite_fused] <= panel_smoothed.P_obs.values[finite_fused] + 1e-6)
print("extend_states_prior_smoothed -> combine_states_priors: no collapse, fusion still sane.")


def mean_over_finite(ssp_prior):
    "Per-state mean P_obs over its finite (disclosed) steps only."
    p = ssp_prior.P_obs.values
    return np.where(np.isfinite(p), p, np.nan).mean(axis=0)


summary = pd.DataFrame(
    {
        "vendor (nearest)": mean_over_finite(vendor),
        "vendor (smoothed)": mean_over_finite(vendor_smoothed),
        "panel (nearest)": mean_over_finite(panel),
        "panel (smoothed)": mean_over_finite(panel_smoothed),
        "fused (nearest)": mean_over_finite(fused),
        "fused (smoothed)": mean_over_finite(fused_smoothed),
    },
    index=pd.Index(state_labels, name="state"),
)
summary.round(4)  # P_obs (mean over finite steps), both extension methods side by side


The fused variance is at or below the tighter of the two inputs at every finite step (never wider),
and `level` -- disclosed by neither study -- stays `NaN`/`inf` throughout, since `inf` is zero
precision and drops out of the sum entirely. This holds for **both** extension methods: the
assertions above additionally confirm the smoothed extension never collapses toward zero before a
channel's first anchor (issue #69), the failure mode this comparison guards against.


In [ ]:
def prior_samples(ssp_prior, n_draws=2000, seed=0):
    """Draw Monte-Carlo samples from a prior's per-step Gaussian marginal.

    Parameters
    ----------
    ssp_prior : xr.Dataset or SspPrior
        Prior with ``a_obs`` / ``P_obs`` over ``(time, state)``.
    n_draws : int, optional
        Number of Monte Carlo draws, by default 2000.
    seed : int, optional
        RNG seed, by default 0.

    Returns
    -------
    np.ndarray, shape (n_draws, n_steps, n_states)
        Draws from ``N(a_obs, P_obs)`` per step; undisclosed (``inf``-variance)
        steps are set to ``NaN`` so the ribbon shows a gap there.
    """
    rng = np.random.default_rng(seed)
    a = np.asarray(ssp_prior["a_obs"].values, dtype=float)
    p = np.asarray(ssp_prior["P_obs"].values, dtype=float)
    draws = a[None] + np.sqrt(p)[None] * rng.standard_normal((n_draws, *a.shape))
    draws[:, ~np.isfinite(p)] = np.nan
    return draws


In [ ]:
# extend_states_prior_smoothed first: it is the more powerful choice for a channel disclosed by
# multiple anchors, like tv / search / social here (construct_states_prior(..., n_periods=3,
# n_points=4, ...) gives each its own several disclosures per study) -- see the markdown below for
# why, with the comparison-table numbers as concrete evidence.
posterior_smoothed = {
    "vendor study": prior_samples(vendor_smoothed),
    "internal panel": prior_samples(panel_smoothed),
    "fused": prior_samples(fused_smoothed),
}

fig, axes = plot_states(
    posterior_smoothed,
    dates.values,
    state_labels,
    states_key=list(posterior_smoothed),
    title="Fusing two independent studies of the same channels (extend_states_prior_smoothed)",
    n_cols=2,
    colors={"vendor study": "steelblue", "internal panel": "darkorange", "fused": "darkgreen"},
)
plt.show()


In [ ]:
# Same three series, same colors and layout, fed by extend_states_prior_nearest instead -- the
# cheaper single-anchor heuristic, shown for comparison against the smoothed figure above (full
# visual parity between the two methods, not just the numeric comparison table in section 2.1).
posterior = {
    "vendor study": prior_samples(vendor),
    "internal panel": prior_samples(panel),
    "fused": prior_samples(fused),
}

fig, axes = plot_states(
    posterior,
    dates.values,
    state_labels,
    states_key=list(posterior),
    title="Fusing two independent studies of the same channels (extend_states_prior_nearest)",
    n_cols=2,
    colors={"vendor study": "steelblue", "internal panel": "darkorange", "fused": "darkgreen"},
)
plt.show()


**`extend_states_prior_smoothed` is the more powerful choice for a real multi-anchor channel
like this one.** `tv` / `search` / `social` are each disclosed several times per study
(`construct_states_prior(..., n_periods=3, n_points=4, ...)`), and `_smoothed` fuses *every*
anchor in a channel by inverse-variance weighting -- it blends means continuously between
anchors, is never wider than `_nearest`, and is strictly tighter once a channel carries two or
more. `_nearest` is the cheap heuristic: it snaps each undisclosed step to whichever anchor is
closest and never blends, so its ribbon has a visible discontinuous "tent" kink at the midpoint
between any two anchors. The comparison table above is the concrete evidence: vendor `P_obs` is
0.199 (smoothed) vs. 0.2113 (nearest), and fused is 0.0822 (smoothed) vs. 0.096 (nearest) --
smoothed tighter throughout.

**In the first figure** (`extend_states_prior_smoothed`), the **fused** ribbon (green) sits
inside both inputs wherever they overlap, pulled toward whichever study was tighter at that
step, and is narrower than either everywhere both disclose something. Where only one study
discloses a window, the fused ribbon just matches that one study -- the other contributes zero
precision there. `level` stays empty in all three panels.

**The second figure** (`extend_states_prior_nearest`) repeats this with the cheaper heuristic
feeding `vendor` / `panel` / `fused` instead: the same qualitative fusion story, but every
ribbon is a little wider throughout, and the tent-kink discontinuity is visible in `vendor` and
`panel` between adjacent anchors -- exactly what `_smoothed` avoids by blending instead of
snapping.


### 2.2 Closed-form sanity checks

Three degenerate cases the fusion rule handles explicitly rather than leaving to produce `NaN`, on
tiny hand-built single-state priors (mirroring `tests/test_combine_states_priors.py`).


In [ ]:
def tiny_prior(a, p, n_steps=3):
    "Build a single-state complete prior with constant mean / variance everywhere."
    return SspPrior(
        xr.Dataset(
            {
                "a0": (("state",), np.array([a])),
                "P0": (("state",), np.array([p])),
                "a_obs": (("time", "state"), np.full((n_steps, 1), a)),
                "P_obs": (("time", "state"), np.full((n_steps, 1), p)),
                "positivity": (("state",), np.array([False])),
            },
            coords={"time": np.arange(n_steps), "state": ["x"]},
        )
    )


# (a) Identical priors halve the variance: N(a, P) fused with itself is N(a, P/2).
same = combine_states_priors(tiny_prior(3.0, 0.4), tiny_prior(3.0, 0.4))
print("identical:  a =", same.a_obs.values[0, 0], " P =", same.P_obs.values[0, 0], " (expect a=3.0, P=0.2)")

# (b) A tight prior dominates a loose one.
dominated = combine_states_priors(tiny_prior(10.0, 1e-4), tiny_prior(0.0, 1e4))
print("dominated:  a =", round(float(dominated.a_obs.values[0, 0]), 4), " (expect ~10.0)")

# (c) Fusing with a fully-undisclosed prior passes the informative one through unchanged.
passthrough = combine_states_priors(tiny_prior(5.0, 0.5), tiny_prior(0.0, np.inf))
a_p, P_p = passthrough.a_obs.values[0, 0], passthrough.P_obs.values[0, 0]
print("passthrough: a =", a_p, " P =", P_p, " (expect a=5.0, P=0.5)")


### 2.3 More than two fragments: `functools.reduce`

Precision adds, so fusion is associative -- there's no separate n-ary entry point in the package.
`functools.reduce(combine_states_priors, fragments)` gives the same one-pass closed form regardless of
fold order, and fusing $N$ identical copies of $N(a, P)$ gives $N(a, P/N)$.


In [ ]:
from functools import reduce

fragments = [tiny_prior(3.0, 0.4) for _ in range(4)]
folded = reduce(combine_states_priors, fragments)
print("4 copies of N(3.0, 0.4) fuse to P =", folded.P_obs.values[0, 0], " (expect 0.4 / 4 = 0.1)")

# Fold order is free when every operand is genuinely informative.
a, b, c = tiny_prior(1.0, 2.0), tiny_prior(4.0, 1.0), tiny_prior(2.0, 3.0)
left_fold = combine_states_priors(combine_states_priors(a, b), c)
right_fold = combine_states_priors(a, combine_states_priors(b, c))
print(
    "associative:",
    np.allclose(left_fold.a_obs.values, right_fold.a_obs.values)
    and np.allclose(left_fold.P_obs.values, right_fold.P_obs.values),
)


Three independent fragments `a`, `b`, `c` in, one fused prior out -- drawn as what they actually are,
per-state Gaussian densities, rather than read off as numbers. `combined` is narrower than every one of
`a` / `b` / `c` (precision adds) and its mean sits closest to `b`, the tightest of the three.


In [ ]:
def gaussian_pdf(y, mean, var):
    """Evaluate the 1-D Gaussian density N(y; mean, var) pointwise over `y`."""
    return np.exp(-0.5 * (y - mean) ** 2 / var) / np.sqrt(2 * np.pi * var)


combined = reduce(combine_states_priors, [a, b, c])  # same fold `left_fold` / `right_fold` computed above

fragments = {"a": a, "b": b, "c": c, "combined = reduce(fuse, [a, b, c])": combined}
colors = {
    "a": "steelblue",
    "b": "darkorange",
    "c": "seagreen",
    "combined = reduce(fuse, [a, b, c])": "crimson",
}

means = {label: float(p.a0.values[0]) for label, p in fragments.items()}
variances = {label: float(p.P0.values[0]) for label, p in fragments.items()}
lo = min(m - 3.5 * np.sqrt(variances[label]) for label, m in means.items())
hi = max(m + 3.5 * np.sqrt(variances[label]) for label, m in means.items())
y = np.linspace(lo, hi, 400)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for label, color in colors.items():
    pdf = gaussian_pdf(y, means[label], variances[label])
    is_combined = label.startswith("combined")
    ax.plot(
        y,
        pdf,
        color=color,
        lw=2.5 if is_combined else 1.5,
        ls="--" if not is_combined else "-",
        label=f"{label}:  N({means[label]:.2f}, {variances[label]:.2f})",
    )
    ax.fill_between(y, pdf, color=color, alpha=0.12 if not is_combined else 0.2)
ax.set_xlabel("state value")
ax.set_ylabel("density")
ax.set_title("Three independent priors fused into one, as densities")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 2.4 The contract is enforced, not just documented

`combine_states_priors` compares (never aligns) the `time` / `state` coordinates and requires an
identical `positivity` mask -- xarray's implicit alignment would otherwise quietly intersect mismatched
axes and fill `NaN`.


In [ ]:
linear = tiny_prior(1.0, 1.0)
positivity_mismatch = SspPrior(linear.dataset.assign(positivity=(("state",), np.array([True]))))
try:
    combine_states_priors(linear, positivity_mismatch)
except ValueError as exc:
    print("positivity mismatch:", exc)

shorter = tiny_prior(1.0, 1.0, n_steps=2)
try:
    combine_states_priors(linear, shorter)
except ValueError as exc:
    print("time mismatch:      ", exc)


## Takeaways

- **`SspPrior`** is a validated checkpoint, not a durable type: it enforces the complete-prior contract
  once, at construction, and exposes a read-only facade (`prior["a0"]`, `prior.sizes`, `in`, `len`,
  iteration) over the wrapped dataset. It doesn't survive `xr.merge` (unwrap with `.to_dataset()`
  first) or ordinary xarray ops like `.isel()` (re-promote with `.from_dataset()` after).
  `construct_states_prior`'s raw output is *not* yet a complete prior -- an extension function
  (`extend_states_prior_nearest` / `_smoothed`) or a hand-supplied `a0` / `P0` is what completes it.
- **`combine_states_priors`** fuses two *independent* priors over the same states by inverse-variance
  weighting: precision adds, so the fused variance never widens, an undisclosed operand passes the
  other straight through, and a tight prior dominates a loose one. It is associative
  (`functools.reduce` handles any number of fragments) but not commutative in the fully-degenerate
  corner cases the docstring calls out. Only the four moment variables (`a0`, `P0`, `a_obs`, `P_obs`)
  are fused; everything else -- and any positivity mismatch -- is either taken from the left operand or
  rejected outright.
- **The independence assumption is load-bearing and unchecked.** Fusing two extensions of the *same*
  anchor set double-counts evidence and understates the fused variance; nothing here can detect that,
  it's the caller's to rule out. Fusion is also not a mixture -- see the function's own docstring Notes
  for the distinction (fusion shrinks variance for complementary evidence about one truth; a mixture
  grows it for genuine disagreement between two sources, and is deliberately out of scope here).
